# Week 8 Lab — GJR, EGARCH, t Errors, Volatility Forecasts and VaR

**MANG2074 Financial Econometrics 1**

**Objectives**

- Estimate asymmetric volatility models (GJR-GARCH, EGARCH) and test for leverage effects.
- Estimate GARCH with Student-t errors and interpret the degrees of freedom.
- Produce fixed-window out-of-sample volatility forecasts.
- Compute a 1% one-day parametric VaR from a GARCH forecast.

**Data**

- `../data/currencies.csv` — daily `EUR`, `GBP`, `JPY` dollar rates, 1998–2018.
- `../data/sp500.csv` — daily S&P 500, 1950–2018.


## Setup

Build daily % log returns for all three currencies and the S&P 500.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

fx = pd.read_csv('../data/currencies.csv', index_col=0, parse_dates=True)
rets = pd.DataFrame({cur: 100 * np.log(fx[cur]).diff() for cur in ['EUR', 'GBP', 'JPY']}).dropna()

sp = pd.read_csv('../data/sp500.csv', index_col=0, parse_dates=True)
rsp = (100 * np.log(sp['sp500']).diff()).dropna()
print(rets.describe().round(3))

## Task 1 — Three volatility models for the S&P 500

Fit to `rsp`:

- GARCH(1,1): `arch_model(rsp, p=1, q=1)`
- GJR-GARCH: `arch_model(rsp, p=1, o=1, q=1)`
- EGARCH: `arch_model(rsp, vol='EGARCH', p=1, o=1, q=1)`

Print the GJR summary in full and collect log-likelihood, AIC and BIC for the three models in a table.

In [ ]:
garch_sp = arch_model(rsp, p=1, q=1).fit(disp='off')
# YOUR CODE HERE: gjr_sp, egarch_sp, comparison table


## Task 2 — Is there a leverage effect?

(a) In the GJR model, test $H_0: \gamma = 0$ (read the t-stat/p-value off the summary). What does $\gamma > 0$ mean?
(b) In the EGARCH model, what is the sign of `gamma[1]` and what does it mean there?
(c) Compute the GJR "news impact": variance response to a +1% vs a −1% shock ($\alpha$ vs $\alpha + \gamma$).

In [ ]:
# YOUR CODE HERE


## Task 3 — Leverage in FX?

Fit GJR-GARCH to `rets['JPY']`. Is $\gamma$ significant, and how does its size compare with the S&P? Why might asymmetry be weaker (or differently interpreted) for exchange rates?

In [ ]:
# YOUR CODE HERE


## Task 4 — Student-t errors for the yen

Fit GARCH(1,1) to `rets['JPY']` with `dist='t'`. Report $\hat\nu$ (`nu`) and compare log-likelihoods with the normal-error model. What does a $\nu$ near 3 say about FX tail risk?

In [ ]:
gjpy_n = arch_model(rets['JPY'], p=1, q=1).fit(disp='off')
# YOUR CODE HERE: gjpy_t with dist='t'; compare loglikelihood and nu


## Task 5 — Conditional volatility plots

Fit GARCH(1,1)-t to each of the three currencies; plot the three conditional volatility series on one figure (annualise: multiply by $\sqrt{252}$). Which currency was riskiest, when?

In [ ]:
# YOUR CODE HERE


## Task 6 — Fixed-window volatility forecasts

Estimate GARCH(1,1) on the yen using only data to end-2015 (`fit(last_obs='2016-01-01')`), then produce one-step-ahead variance forecasts for 2016 onwards:

```python
fc = res.forecast(horizon=1, start='2016-01-01', reindex=False)
```

Plot forecast volatility ($\sqrt{h.1}$) against realised absolute returns. Does the forecast track turbulence?

In [ ]:
am = arch_model(rets['JPY'], p=1, q=1)
res_fixed = am.fit(last_obs='2016-01-01', disp='off')
# YOUR CODE HERE: forecast, plot vs |returns|


## Task 7 — 1% one-day parametric VaR

Using the full-sample yen GARCH-t model: forecast tomorrow's volatility (`forecast(horizon=1)`), then compute the 1% one-day VaR for a **£1m long-yen position**, under (a) normal innovations ($q_{0.01}=-2.326$) and (b) Student-t innovations ($q = t_{0.01}(\hat\nu)\sqrt{(\hat\nu-2)/\hat\nu}$).

$$VaR = -(\hat\mu + q\,\hat\sigma_{T+1}) \times \frac{£1{,}000{,}000}{100}$$

In [ ]:
# YOUR CODE HERE


## Task 8 — Risk report

Write 5–8 sentences as a junior risk analyst: which volatility model would you deploy for the equity desk and which for the FX desk, and why? Report the VaR numbers and explain why the t-based VaR differs from the normal one.

*YOUR ANSWER HERE*